In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [3]:
df = pd.read_csv("/content/loan_approval_dataset.csv")


In [4]:
df.columns = df.columns.str.strip()


In [5]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (4269, 13)
   loan_id  no_of_dependents     education self_employed  income_annum  \
0        1                 2      Graduate            No       9600000   
1        2                 0  Not Graduate           Yes       4100000   
2        3                 3      Graduate            No       9100000   
3        4                 3      Graduate            No       8200000   
4        5                 5  Not Graduate           Yes       9800000   

   loan_amount  loan_term  cibil_score  residential_assets_value  \
0     29900000         12          778                   2400000   
1     12200000          8          417                   2700000   
2     29700000         20          506                   7100000   
3     30700000          8          467                  18200000   
4     24200000         20          382                  12400000   

   commercial_assets_value  luxury_assets_value  bank_asset_value loan_status  
0                 17600000             2

In [6]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64


In [7]:
print("Number of duplicated rows:", df.duplicated().sum())


Number of duplicated rows: 0


In [8]:
df = df.drop_duplicates()


In [9]:
df = df.drop("loan_id", axis=1)


In [10]:
df["education"] = df["education"].map({
    "Graduate": 1,
    "Not Graduate": 0
})

In [11]:
df["self_employed"] = df["self_employed"].map({
    "Yes": 1,
    "No": 0
})


In [12]:
df["loan_status"] = df["loan_status"].map({
    "Approved": 1,
    "Rejected": 0
})

In [13]:
X = df.drop("loan_status", axis=1)

In [14]:
y = df["loan_status"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


In [18]:
logistic_model.fit(X_train_scaled, y_train)
logistic_pred = logistic_model.predict(X_test_scaled)

In [19]:
logistic_accuracy = accuracy_score(
    y_test,
    logistic_pred
)


In [20]:
ridge_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    random_state=42
)

In [21]:
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)
ridge_accuracy = accuracy_score(
    y_test,
    ridge_pred
)

In [22]:
lasso_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

In [23]:
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)
lasso_accuracy = accuracy_score(
    y_test,
    lasso_pred
)

In [24]:
dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [25]:
dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


In [26]:
dt_grid.fit(X_train, y_train)
best_dt = dt_grid.best_estimator_
dt_grid_pred = best_dt.predict(X_test)
dt_grid_accuracy = accuracy_score(
    y_test,
    dt_grid_pred
)

In [27]:
print("Best Decision Tree Parameters:")
print(dt_grid.best_params_)


Best Decision Tree Parameters:
{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 10}


In [28]:
rf_model = RandomForestClassifier(
    random_state=42
)

In [29]:
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

In [30]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [31]:
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [32]:
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
rf_grid_pred = best_rf.predict(X_test)

In [33]:
rf_grid_accuracy = accuracy_score(
    y_test,
    rf_grid_pred
)

In [34]:
print("\nBest Random Forest Parameters:")
print(rf_grid.best_params_)


Best Random Forest Parameters:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [35]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy:", dt_accuracy)

Decision Tree Accuracy: 0.9812646370023419


In [36]:
dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring="accuracy"
)

dt_grid.fit(X_train, y_train)

dt_best = dt_grid.best_estimator_

dt_grid_pred = dt_best.predict(X_test)

dt_grid_accuracy = accuracy_score(y_test, dt_grid_pred)

print("Best Parameters:", dt_grid.best_params_)
print("Decision Tree After Grid Search:", dt_grid_accuracy)

Best Parameters: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 10}
Decision Tree After Grid Search: 0.9800936768149883


In [39]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Ridge",
        "Lasso",
        "Decision Tree Before Grid Search",
        "Decision Tree After Grid Search",
        "Random Forest Before Grid Search",
        "Random Forest After Grid Search"
    ],

    "Accuracy": [
logistic_accuracy,
                  ridge_accuracy,
        lasso_accuracy,
        dt_accuracy,
        dt_grid_accuracy,
        rf_accuracy,
        rf_grid_accuracy
    ]
})

results["Accuracy"] = (results["Accuracy"] * 100).round(2)

results

,Model,Accuracy
0,Logistic Regression,91.33
1,Ridge,91.33
2,Lasso,91.33
3,Decision Tree Before Grid Search,98.13
4,Decision Tree After Grid Search,98.01
5,Random Forest Before Grid Search,98.01
6,Random Forest After Grid Search,98.01


In [45]:
results["Accuracy (%)"] = results["Accuracy"] * 100
results["Accuracy (%)"] = results["Accuracy (%)"].round(2)
results = results.drop("Accuracy", axis=1)

In [43]:
print(
    "Best Model:",
    best_model["Model"]
)

Best Model: Decision Tree Before Grid Search


In [46]:
print(results)

                              Model  Accuracy (%)
0               Logistic Regression        9133.0
1                             Ridge        9133.0
2                             Lasso        9133.0
3  Decision Tree Before Grid Search        9813.0
4   Decision Tree After Grid Search        9801.0
5  Random Forest Before Grid Search        9801.0
6   Random Forest After Grid Search        9801.0


In [48]:
best_model = results.loc[
    results["Accuracy (%)"].idxmax()
]

In [49]:
print(best_model)

Model           Decision Tree Before Grid Search
Accuracy (%)                              9813.0
Name: 3, dtype: object
